In [ ]:
%reset -f
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [4]:
PermAccuracy = pd.read_csv(r'D:\research\HCP_parcellation_Lsym-AVR\CamCan\KRR parcel size to cognition\model perm test\all_input_perm_accuracy.csv')
TrueAccuracy = pd.read_csv(r"D:\research\HCP_parcellation_Lsym-AVR\CamCan\KRR parcel size to cognition\evaluation\All cogTasks.csv")
TrueAccuracyMean = pd.DataFrame(TrueAccuracy.mean())
# CogTasks = ['Fluid intelligence','Emotion recognition','Picture priming','Face recognition','Famous faces','Motor learning']
CogTasks = ['Gf','EER','PPR','FR','FF','ML']
NPerm = 1000

In [5]:
Pvalue = []
for i in range(len(TrueAccuracyMean)):
    CurrP = (1+(PermAccuracy.iloc[:,i]>TrueAccuracyMean.iloc[i,0]).sum())/(1+NPerm)
    Pvalue.append(CurrP)
TrueAccuracyMean['Pvalue'] = Pvalue

In [6]:
PermAccuracyLong = PermAccuracy.melt()

In [ ]:
sns.set_theme(style="white", rc={"axes.facecolor": (0, 0, 0, 0), "axes.linewidth": 1})
plt.rcParams['font.family'] = ['Times New Roman','SimSun']
# Task order (should align with CogTasks later)
task_order = PermAccuracyLong["variable"].unique()

# Assign a color to each task (you can change to your preferred palette)
palette = dict(zip(task_order, sns.color_palette("husl", n_colors=len(task_order))))
# You can also customize:
# palette = {
#     task_order[0]: "#4C78A8",
#     task_order[1]: "#F58518",
#     task_order[2]: "#54A24B",
#     task_order[3]: "#E45756",
#     task_order[4]: "#72B7B2",
#     task_order[5]: "#B279A2",
# }

g = sns.FacetGrid(
    PermAccuracyLong,
    row="variable",
    row_order=task_order,
    hue="variable",
    palette=palette,
    aspect=9,
    height=1.2
)

g.figure.set_size_inches(3, 2.9)
g.figure.set_dpi(300)

# Fill layer: each task filled with its own color
g.map_dataframe(sns.kdeplot, x="value", fill=True, alpha=0.95, linewidth=0)

# Outline layer: draw black border lines for each subplot individually
for ax, var in zip(g.axes.flat, task_order):
    sub = PermAccuracyLong[PermAccuracyLong["variable"] == var]
    sns.kdeplot(
        data=sub, x="value",
        fill=False, color="black", linewidth=1,
        ax=ax, legend=False
    )

ax = g.axes.flat[0]
LeftXlim = ax.get_xlim()
g.set(xlim=(LeftXlim[0],0.3))

for i,ax in enumerate(g.axes.flat):
    ax.text(0, 0.2, CogTasks[i], color='black', fontsize=8,
                ha="left", va="center", transform=ax.transAxes)
    ax.axvline(TrueAccuracyMean[0][i],color='r',linewidth=2)
    if TrueAccuracyMean['Pvalue'][i]<0.001:
        ax.text(TrueAccuracyMean[0][i]+0.01,2,'P<0.001',color='black', fontsize=7,ha="left", va="center")
    else:
        p=TrueAccuracyMean['Pvalue'][i]
        ax.text(TrueAccuracyMean[0][i]+0.01,2,f'P={p:.3f}',color='black', fontsize=7,ha="left", va="center")

g.figure.subplots_adjust(hspace=.1)
g.set_titles("")
g.set(yticks=[], ylabel='')
g.set_xlabels("Accuracy (Pearson correlation)",fontsize=10)
g.tick_params(axis='x',labelsize=8)
g.despine(left=True)
plt.subplots_adjust(left=.02,right=.92,bottom=.16)
plt.savefig('Ridgeplot.svg',format='svg')

In [ ]:
# Boxplot data: take the first few columns of TrueAccuracy that correspond to CogTasks
# If your task columns are not the first few, replace TrueAccuracy.iloc[:, :len(CogTasks)] with explicit column names
BoxData = TrueAccuracy.iloc[:, :len(CogTasks)].copy()
BoxData.columns = CogTasks
BoxLong = BoxData.melt(var_name="Cognitive task", value_name="Accuracy")
plt.rcParams["xtick.bottom"]=True
plt.rcParams["ytick.left"]=True
# Color scheme consistent with the ridge plot (husl)
palette_box = dict(zip(CogTasks, sns.color_palette("husl", n_colors=len(CogTasks))))

fig, ax = plt.subplots(figsize=(3.2, 2.9), dpi=300)

sns.boxplot(
    data=BoxLong,
    x="Cognitive task",
    y="Accuracy",
    order=CogTasks,
    palette=palette_box,
    width=0.6,
    linewidth=1,
    whis=(2.5, 97.5),          # Whiskers extend to 2.5% and 97.5%
    showfliers=True,            # Show outliers beyond the whiskers
    flierprops={
        "marker": "o",
        "markerfacecolor": "black",
        "markeredgecolor": "black",
        "markersize": 1,
        "alpha": 1
    },
    boxprops={"edgecolor": "black", "linewidth": 1},
    whiskerprops={"color": "black", "linewidth": 1},
    capprops={"color": "black", "linewidth": 1},
    medianprops={"color": "black", "linewidth": 1},
    ax=ax
)

# Axis labels
ax.set_ylabel("Accuracy (Pearson correlation)", fontsize=10)
ax.set_xlabel("Cognitive task", fontsize=10)

# Line widths and style consistent with the ridge plot
ax.tick_params(axis="x", labelsize=8, width=1, length=3, direction='out')
ax.tick_params(axis="y", labelsize=8, width=1, length=3, direction='out')
ax.spines["bottom"].set_linewidth(1)
ax.spines["left"].set_linewidth(1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("Boxplot_TrueAccuracy.svg", format="svg")

In [ ]:
plt.rcParams["xtick.bottom"]